# Notebook 02 — Limpieza, Calidad y Transformación Distribuida (Capa Silver)
**Proyecto:** Implementación de Soluciones BI, Big Data y Experimentación — Unity Perú SAC
**Arquitectura:** Medallion Lakehouse (Bronze $\rightarrow$ Silver)

### Operaciones de DataFrames PySpark demostradas
- `select()`, `withColumnRenamed()`, `withColumn()`: normalización a `snake_case` y derivación de atributos temporales (`anio`, `mes`, `hora`) y tramos de velocidad FIFO.
- `na.drop()`, `na.fill()`: descarte de registros con atributos obligatorios nulos (RQ-01) e imputación semántica de llaves opcionales (`SIN_PROYECTO`).
- `filter()`, `dropDuplicates()`: cumplimiento estricto de cantidades positivas (RQ-02) y unicidad de eventos (RQ-04).
- `join()` (inner y left): validación referencial contra catálogo (RQ-03) y desnormalización controlada hacia tablas Silver particionadas por `mes`.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("UnityPeru_02_Silver").getOrCreate()
spark.sql("CREATE SCHEMA IF NOT EXISTS unity_silver")
FORMATO_TABLA = "delta"

df_mov_raw = spark.table("unity_bronze.movimientos_raw")
df_prod_raw = spark.table("unity_bronze.productos_raw")
df_tipo_raw = spark.table("unity_bronze.tipos_movimiento_raw")
df_proy_raw = spark.table("unity_bronze.proyectos_raw")
df_plan_raw = spark.table("unity_bronze.plan_demanda_raw")
df_sen_raw  = spark.table("unity_bronze.sensores_iot_raw")
df_con_raw  = spark.table("unity_bronze.conteos_ciclicos_raw")


## 1. Limpieza, Renombrado y Enriquecimiento de Catálogo y Movimientos (`silver_movimientos`)


In [ ]:
# 1.1 Estandarización del catálogo tecnológico
df_prod_silver = (
    df_prod_raw.select(
        "Id_Producto", "Cod_Componente", "Descripcion", "Categoria",
        "Marca", "Modelo", "Criticidad", "Stock_Minimo", "Precio_Unitario"
    )
    .withColumnRenamed("Cod_Componente", "codigo_componente")
    .withColumnRenamed("Categoria", "categoria")
    .withColumnRenamed("Criticidad", "criticidad")
    .withColumnRenamed("Stock_Minimo", "stock_minimo")
    .withColumnRenamed("Precio_Unitario", "precio_unitario")
    .na.drop(subset=["Id_Producto", "codigo_componente"])
    .dropDuplicates(["codigo_componente"])
    .withColumn("categoria", F.trim(F.col("categoria")))
    .withColumn("criticidad", F.upper(F.trim(F.col("criticidad"))))
)

# 1.2 Aplicación de reglas RQ-01..RQ-04 y enriquecimiento en Capa Silver
df_mov_silver = (
    df_mov_raw
    .na.drop(subset=["Id_Movimiento_NK", "Id_Fecha", "Id_Producto", "Id_TipoMovimiento", "Cantidad"])
    .filter(F.col("Cantidad") > 0)
    .dropDuplicates(["Id_Movimiento_NK"])
    .withColumnRenamed("Id_Movimiento_NK", "id_movimiento")
    .withColumnRenamed("Cantidad", "cantidad")
    .withColumnRenamed("Costo_Total", "costo_total")
    .withColumnRenamed("Dias_Permanencia", "dias_permanencia")
    .withColumn("fecha_movimiento", F.to_date(F.col("Fecha").cast("string")))
    .withColumn("anio", F.year("fecha_movimiento"))
    .withColumn("mes", F.month("fecha_movimiento"))
    .na.fill({"Id_Proyecto": -1, "Id_Proveedor": -1})
    .join(
        df_prod_silver.select("Id_Producto", "codigo_componente", "categoria", "criticidad", "precio_unitario", "stock_minimo"),
        on="Id_Producto", how="inner"
    )
    .join(
        df_tipo_raw.select(
            "Id_TipoMovimiento",
            F.col("Codigo").alias("tipo_movimiento"),
            F.col("Es_Entrada").alias("es_entrada"),
            F.col("Es_Salida_Real").alias("es_salida_real")
        ),
        on="Id_TipoMovimiento", how="inner"
    )
    .join(
        df_proy_raw.select(
            "Id_Proyecto",
            F.col("Codigo_Proyecto").alias("codigo_proyecto"),
            F.col("Nombre_Proyecto").alias("nombre_proyecto")
        ),
        on="Id_Proyecto", how="left"
    )
    .na.fill({"codigo_proyecto": "SIN_PROYECTO", "nombre_proyecto": "Operación general de almacén"})
    .withColumn(
        "tramo_permanencia",
        F.when(F.col("es_entrada") == 1, F.lit("INGRESO_LOTE"))
        .when(F.col("dias_permanencia") <= 3, F.lit("RAPIDO_0_3D"))
        .when(F.col("dias_permanencia") <= 7, F.lit("NORMAL_4_7D"))
        .when(F.col("dias_permanencia") <= 10, F.lit("MODERADO_8_10D"))
        .otherwise(F.lit("RETENIDO_MAS_10D"))
    )
)

df_prod_silver.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_silver.productos")
(
    df_mov_silver.write.format(FORMATO_TABLA)
    .mode("overwrite")
    .partitionBy("mes")
    .saveAsTable("unity_silver.movimientos")
)

df_mov_silver.groupBy("tramo_permanencia").agg(
    F.count("*").alias("movimientos"), F.sum("cantidad").alias("unidades")
).orderBy(F.desc("movimientos")).show()


## 2. Preparación de Telemetría IoT, Conteos Cíclicos y Plan Mensual en Capa Silver


In [ ]:
df_sensores_silver = (
    df_sen_raw
    .na.drop(subset=["timestamp", "sensor_id", "codigo_ubicacion", "temperatura_c", "humedad_pct"])
    .dropDuplicates(["timestamp", "sensor_id"])
    .withColumn("ts_lectura", F.to_timestamp("timestamp"))
    .withColumn("fecha", F.to_date("ts_lectura"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("hora", F.hour("ts_lectura"))
    .withColumn(
        "fuera_rango",
        F.when(
            (F.col("temperatura_c") < 18.0) | (F.col("temperatura_c") > 27.0) |
            (F.col("humedad_pct") < 40.0) | (F.col("humedad_pct") > 60.0), 1
        ).otherwise(0)
    )
)

df_conteos_silver = (
    df_con_raw
    .na.drop(subset=["fecha", "cod_componente", "stock_sistema", "stock_fisico"])
    .withColumnRenamed("cod_componente", "codigo_componente")
    .withColumn("fecha_conteo", F.to_date("fecha"))
    .withColumn("mes", F.month("fecha_conteo"))
    .withColumn("diferencia_calc", F.col("stock_fisico") - F.col("stock_sistema"))
    .withColumn("denominador", F.greatest(F.col("stock_sistema"), F.col("stock_fisico"), F.lit(1)))
    .withColumn("exactitud_linea", F.round(1.0 - (F.abs(F.col("diferencia_calc")) / F.col("denominador")), 4))
)

df_plan_silver = (
    df_plan_raw
    .na.drop(subset=["Anio", "Mes", "Categoria", "Entradas_Plan", "Salidas_Plan"])
    .withColumnRenamed("Anio", "anio")
    .withColumnRenamed("Mes", "mes")
    .withColumnRenamed("Categoria", "categoria")
    .withColumnRenamed("Entradas_Plan", "entradas_plan")
    .withColumnRenamed("Salidas_Plan", "salidas_plan")
    .filter((F.col("entradas_plan") >= 0) & (F.col("salidas_plan") >= 0))
)

df_sensores_silver.write.format(FORMATO_TABLA).mode("overwrite").partitionBy("mes").saveAsTable("unity_silver.sensores_iot")
df_conteos_silver.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_silver.conteos_ciclicos")
df_plan_silver.write.format(FORMATO_TABLA).mode("overwrite").saveAsTable("unity_silver.plan_demanda")

print(f"Silver completado -> Movimientos: {df_mov_silver.count():,} | Sensores: {df_sensores_silver.count():,} | Conteos: {df_conteos_silver.count()}")
